In [96]:
import random
import pandas as pd
import numpy as np
# from collections import defaultdict
from collections import Counter, defaultdict

In [97]:
POPULASI = 50
VIOLATION_COST = 100
ITERATION = 2000

In [98]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')

# DICT

In [119]:
# =========================================================
# Hari
hariId = {
"Senin": 1,
"Selasa": 2,
"Rabu": 3,
"Kamis": 4,
"Jumat": 5,
}
# hariId

# ========================================================
# mencari slot tiap per hari
# key = hariId, value = jumlah slot integer
# contoh output: {1: 8, 2: 8, 3: 8, 4: 7, 5: 5}

slotPerHari = {} # dictionary kosongan
for _, row in slot_df.iterrows():
    hari = row["hari"] # ambil kolom hari saja

    if hari not in slotPerHari:
        slotPerHari[hari] = 1
    else:
        slotPerHari[hari] += 1

slotPerHari = {hariId[k]: v for k,v in slotPerHari.items()}
# print(slotPerHari)


# ========================================================
# guru dan nama
# key = guru_id, value = nama guru
guruPengajar = dict(
    zip(guru_df['guru_id'], guru_df['nama_guru'])
)
# guruPengajar

# ========================================================
# mapping nama kelas dan tingkatan
# ada 27 kelas, contoh output: {1: [{'tingkatan': 7, 'nama_kelas': '7A'}], 2: [{'tingkatan': 7, 'nama_kelas': '7B'}]}
kelasDanTingkatan = defaultdict(list)
for _, row in kelas_df.iterrows():
    kelasDanTingkatan[row['kelas_id']].append({
        'tingkatan': row['tingkatan'],
        'nama_kelas': row['nama_kelas']
    })
kelasDanTingkatan = dict(kelasDanTingkatan)
# print(kelasDanTingkatan)

# ========================================================
# mapping nama mapel dan id
# key = mapel_id, value = nama mapel
namaMapelDanId = dict(
    zip(mapel_df['mapel_id'], mapel_df['nama_mapel'])
)
# namaMapelDanId

# =========================================================
# mencari jam per minggu tiap mape
# key = mapel_id, value = jam per minggu integerl
jamPerMingguMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['jam_per_minggu'])
)
# print(jamPerMingguMapel)

# =========================================================
# mapel id dan hari MGMP
# contoh output: {1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 3, 7: 1, 8: 1, 9: 3, 10: 4, 11: 5, 12: 3, 13: 2}
mgmpMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['MGMP'])
)
mgmpMapel = {mapel_id: hariId[hari] for mapel_id, hari in mgmpMapel.items()}
# print(mgmpMapel)
# =========================================================
# batas siang dan batas MGMP
# key = hariId value = slot ke berapa dalam hari tersebut
batasSiang = {1: 5, 2: 5, 3: 4, 4: 5, 5: 4}
batasMGMP = {1: 2, 2: 2, 3: 2, 4: 2, 5: 1}
# =========================================================
# durasi guru mengajar
# key = guru value = list of dict {mapel_id, tingkatan, durasi}
durasiGuruMengajar = defaultdict(list)

for _, row in relasi_guru_mapel_df.iterrows():
    durasiGuruMengajar[row["guru_id"]].append({
        "mapel_id": row["mapel_id"],
        "tingkatan": row["tingkatan"],
        "durasi": row["durasi"]
    })
durasiGuruMengajar = dict(durasiGuruMengajar)
# durasiGuruMengajar

# INDIVIDU

In [120]:
# memecah jam_per_minggu menjadi blok yang bisa didistribusikan
def blokDistribusi(jam):
    if jam == 2:
        return [2]
    if jam == 3:
        return [3]
    if jam == 4:
        return [2,2]
    if jam == 5:
        return [2,3]
    
    return [jam]

In [121]:
# mengambil guru berdsakan mapel dan tingaktan
def ambilGuruValid(mapel_id, tingkatan):
    listGuru = []

    for guru_id, relasiList in durasiGuruMengajar.items():
        for relasi in relasiList:
            if relasi["mapel_id"] == mapel_id and relasi["tingkatan"] == tingkatan:
                listGuru.append(guru_id)
    return listGuru

In [122]:
# membuat jadwal kosongan dulu
def jadwalKosongan():
    jadwal = {}
    for hari in slotPerHari.keys():
        jadwal[hari] = []

    return jadwal

In [123]:
def slotTersedia(jadwal_kelas, hari, durasi):
    slotTerpakai = 0

    for event in jadwal_kelas[hari]:
        slotTerpakai += event["durasi"]
    
    if slotTerpakai + durasi <= slotPerHari[hari]:
        return True
    
    return False

In [124]:
def putEvent(jadwal_kelas, event):

    listHari = list(slotPerHari.keys())

    random.shuffle(listHari)

    for hari in listHari:

        if slotTersedia(jadwal_kelas, hari, event["durasi"]):
            jadwal_kelas[hari].append(event)
            
            return True
        
    return False

In [125]:
def perluasBlok(mapel_id, guru_id, durasi):
    slot = []

    for _ in range(durasi):
        slot.append({
            "mapel": mapel_id,
            "guru": guru_id
        })
    return slot

In [126]:
def generatePerKelas(tingkatan):

    pilihan = []

    for mapel_id, jam in jamPerMingguMapel.items():

        blok = blokDistribusi(jam)

        guruValid = ambilGuruValid(mapel_id, tingkatan)

        if not guruValid:
            continue

        guru = random.choice(guruValid)

        for durasi in blok:

            slot = perluasBlok(mapel_id, guru, durasi)

            pilihan.extend(slot)

    random.shuffle(pilihan)

    return pilihan

In [127]:
def individuConstruct(kelas_id):

    tingkatan = kelasDanTingkatan[kelas_id][0]["tingkatan"]

    slots = generatePerKelas(tingkatan)

    jadwal = {}

    index = 0

    for hari in slotPerHari:

        jumlahSlot = slotPerHari[hari]

        jadwal[hari] = slots[index:index+jumlahSlot]

        index += jumlahSlot

    return jadwal

In [128]:
def individuTrigger():
    individu = {}

    for kelas_id in kelasDanTingkatan.keys():

        jadwalKelas = individuConstruct(kelas_id)

        individu[kelas_id] = jadwalKelas 

    return individu

In [129]:
def populasiContruct(POPULASI):
    populasi = []

    for _ in range(POPULASI):

        individu = individuTrigger()

        populasi.append(individu)

    return populasi

In [130]:
populasiOptimasi = populasiContruct(POPULASI)

In [131]:
populasiOptimasi[0][1]

{1: [{'mapel': 12, 'guru': 51},
  {'mapel': 1, 'guru': 39},
  {'mapel': 3, 'guru': 22},
  {'mapel': 3, 'guru': 22},
  {'mapel': 7, 'guru': 21},
  {'mapel': 3, 'guru': 22},
  {'mapel': 2, 'guru': 45},
  {'mapel': 3, 'guru': 22}],
 2: [{'mapel': 10, 'guru': 15},
  {'mapel': 13, 'guru': 53},
  {'mapel': 5, 'guru': 48},
  {'mapel': 5, 'guru': 48},
  {'mapel': 11, 'guru': 37},
  {'mapel': 7, 'guru': 21},
  {'mapel': 1, 'guru': 39},
  {'mapel': 9, 'guru': 42}],
 3: [{'mapel': 13, 'guru': 53},
  {'mapel': 5, 'guru': 48},
  {'mapel': 4, 'guru': 47},
  {'mapel': 8, 'guru': 42},
  {'mapel': 8, 'guru': 42},
  {'mapel': 9, 'guru': 42},
  {'mapel': 6, 'guru': 2},
  {'mapel': 4, 'guru': 47}],
 4: [{'mapel': 4, 'guru': 47},
  {'mapel': 12, 'guru': 51},
  {'mapel': 11, 'guru': 37},
  {'mapel': 7, 'guru': 21},
  {'mapel': 4, 'guru': 47},
  {'mapel': 6, 'guru': 2},
  {'mapel': 12, 'guru': 51}],
 5: [{'mapel': 2, 'guru': 45},
  {'mapel': 5, 'guru': 48},
  {'mapel': 6, 'guru': 2},
  {'mapel': 3, 'guru': 2

# EVAL

In [138]:
def guruBentrok(individu):
    pelanggaran = 0

    for hari in slotPerHari:
        totalSlot = slotPerHari[hari]

        for slot in range(totalSlot):
            guruMengajar = []

            for kelas in individu:
                if slot < len(individu[kelas][hari]):
                    guru = individu[kelas][hari][slot]["guru"]
                    guruMengajar.append(guru)

                if len(guruMengajar) != len(set(guruMengajar)):
                    pelanggaran += 1
    return pelanggaran

In [139]:
def putBlokMapel(slotHari):
    blok = []

    mapelSekarang = slotHari[0]["mapel"]

    count = 1

    for i in range(1, len(slotHari)):
        if slotHari[i]["mapel"] == mapelSekarang:
            count += 1
        else:
            blok.append((mapelSekarang, count))

            mapelSekarang = slotHari[i]["mapel"]
            count = 1
    
    blok.append((mapelSekarang, count))

    return blok

def distribusiMapel(individu):
    pelanggaran = 0

    for kelas in individu:
        distribusi = {}

        for hari in individu[kelas]:
            blok = putBlokMapel(individu[kelas][hari])

            for mapel, durasi in blok:

                if mapel not in distribusi:
                    distribusi[mapel] = []

                    distribusi[mapel].append(durasi)
        for mapel in distribusi:
            jam = jamPerMingguMapel[mapel]

            if jam == 2:
                if distribusi[mapel] != [2]:
                    pelanggaran += 1
            elif jam == 3:
                if distribusi[mapel] != [3]:
                    pelanggaran += 1
            elif jam == 4:
                if sorted(distribusi[mapel]) != [2,2]:
                    pelanggaran += 1
            elif jam == 5:
                if sorted(distribusi[mapel]) != [2,3]:
                    pelanggaran += 1
    return pelanggaran

In [140]:
def mapelSiang(individu):
    pelanggaran = 0

    for kelas in individu:

        for hari in individu[kelas]:
            batas = batasSiang[hari]

            for slot in range(len(individu[kelas][hari])):
                mapel = individu[kelas][hari][slot]["mapel"]

                # if mapel == 8 and slot >= batas:
                if mapel == 8 and slot > batas:

                    pelanggaran += 1
    return pelanggaran

In [141]:
def durasiGuru(individu):
    pelanggaran = 0

    loadGuru = {}

    for kelas in individu:
        for hari in individu[kelas]:

            for slot in individu[kelas][hari]:

                guru = slot["guru"]

                if guru not in loadGuru:
                    loadGuru[guru] = 0

                loadGuru[guru] += 1
    for guru in loadGuru:
        if loadGuru[guru] > 40:
            pelanggaran += loadGuru[guru] - 40
    return pelanggaran

In [142]:
def waktuMGMP(individu):
    pelanggaran = 0

    for kelas in individu:

        for hari in individu[kelas]:
            for slot in range(len(individu[kelas][hari])):
                mapel = individu[kelas][hari][slot]["mapel"]

                if mapel in mgmpMapel:
                    hariMGMP = mgmpMapel[mapel]

                    if hari == hariMGMP:
                        if slot > batasMGMP[hari]:
                            pelanggaran += 1

    return pelanggaran

In [143]:
def evaluasiIndividu(individu):

    pelanggaran = 0

    pelanggaran += guruBentrok(individu)

    pelanggaran += distribusiMapel(individu)

    pelanggaran += mapelSiang(individu)

    pelanggaran += durasiGuru(individu)

    pelanggaran += waktuMGMP(individu)

    return pelanggaran